### 1. Regression-mixture clustering of shielding and shift

Purpose: Partition the shielding-versus-shift relationship into two regression-driven regimes without altering the clustering logic.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 1. Regression-mixture clustering of shielding and shift
# Purpose: Partition the shielding-versus-shift relationship into two regression-driven regimes without altering the clustering logic.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.mixture import GaussianMixture

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path('.') .resolve()
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'

DATA_PATH = PROCESSED_DIR / 'cleaned_shift_data_with_shield.csv'
df = pd.read_csv(DATA_PATH)

df_use = (
    df.replace([np.inf, -np.inf], np.nan)
      .dropna(subset=['shielding_value', 'shift_value'])
      .reset_index(drop=True)
)

x = df_use['shielding_value'].astype(float).values.reshape(-1, 1)
y = df_use['shift_value'].astype(float).values

print_section('Input summary')
print(f'Samples: {len(df_use)}')

SEED = 42
K = 2
MAX_ITERS = 200
TOL = 1e-6
MIN_SIGMA = 1e-1

XY = np.column_stack([x.ravel(), y])
init_gmm = GaussianMixture(n_components=K, random_state=SEED, n_init=10)
init_gmm.fit(XY)
resp = init_gmm.predict_proba(XY)

models = [LinearRegression() for _ in range(K)]
sigmas = np.ones(K) * np.std(y)
pis = resp.mean(axis=0)

def logsumexp(a, axis=1):
    amax = np.max(a, axis=axis, keepdims=True)
    return (amax + np.log(np.sum(np.exp(a - amax), axis=axis, keepdims=True))).squeeze()

prev_ll = None

for _ in range(MAX_ITERS):
    for k in range(K):
        w = resp[:, k]
        models[k].fit(x, y, sample_weight=w)
        yhat = models[k].predict(x)
        var = np.sum(w * (y - yhat) ** 2) / (w.sum() + 1e-12)
        sigmas[k] = float(np.sqrt(max(var, MIN_SIGMA ** 2)))

    pis = resp.mean(axis=0)
    pis = pis / pis.sum()

    logp = np.zeros((len(y), K))
    for k in range(K):
        yhat = models[k].predict(x)
        r = y - yhat
        s = sigmas[k]
        logp[:, k] = np.log(pis[k]) - 0.5 * np.log(2 * np.pi * s * s) - 0.5 * (r * r) / (s * s)

    ll = np.sum(logsumexp(logp, axis=1))
    logp -= logsumexp(logp, axis=1)[:, None]
    resp = np.exp(logp)

    if prev_ll is not None and abs(ll - prev_ll) < TOL:
        break
    prev_ll = ll

labels = resp.argmax(axis=1)
conf = resp.max(axis=1)

slope0 = float(models[0].coef_[0])
slope1 = float(models[1].coef_[0])

if slope0 < slope1:
    models[0], models[1] = models[1], models[0]
    sigmas[0], sigmas[1] = sigmas[1], sigmas[0]
    labels = 1 - labels
    conf = 1 - conf
    print('Clusters relabeled: larger slope is now Cluster 0')

print(f'Slope Cluster 0: {models[0].coef_[0]:.4f}')
print(f'Slope Cluster 1: {models[1].coef_[0]:.4f}')

df_out = df_use.copy()
df_out['reg_cluster'] = labels
df_out['reg_prob'] = conf

apply_plot_style()
fig, ax = create_single_axis_figure('tall')

for k in [0, 1]:
    mask = labels == k
    ax.scatter(x[mask], y[mask], s=18, alpha=0.75, label=f'Cluster {k}')

x_line = np.linspace(x.min(), x.max(), 200).reshape(-1, 1)
for k in [0, 1]:
    ax.plot(x_line, models[k].predict(x_line), linewidth=2)

a0 = float(models[0].coef_[0])
b0 = float(models[0].intercept_)
a1 = float(models[1].coef_[0])
b1 = float(models[1].intercept_)

text = (
    f'Cluster 0: y = {a0:.4f}x + {b0:.2f}, sigma={sigmas[0]:.2f}\n'
    f'Cluster 1: y = {a1:.4f}x + {b1:.2f}, sigma={sigmas[1]:.2f}'
)
ax.text(0.02, 0.02, text, transform=ax.transAxes, va='bottom', fontsize=10)

ax.set_title('Regression-mixture clustering of shielding and chemical shift')
ax.set_xlabel('Magnetic shielding value (a.u.)')
ax.set_ylabel(r'$^{19}$F NMR chemical shift (ppm)')
style_axes(ax)
ax.legend(frameon=False)

fig.tight_layout()
plt.show()

summary = (
    df_out.groupby('reg_cluster')
    .agg(
        n=('reg_cluster', 'count'),
        shielding_mean=('shielding_value', 'mean'),
        shift_mean=('shift_value', 'mean'),
        mean_conf=('reg_prob', 'mean'),
    )
    .reset_index()
)

print_section('Cluster summary')
display(prepare_display_table(summary))


### 2. Priority band assignment around the Cluster 1 line

Purpose: Define the prioritized subset by applying a residual band around the selected regression line.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 2. Priority band assignment around the Cluster 1 line
# Purpose: Define the prioritized subset by applying a residual band around the selected regression line.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
TAB_DIR = RESULTS_DIR / 'tables'
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

apply_plot_style()

if 'df_out' not in globals():
    raise RuntimeError('df_out not found. Please run the regression-mixture clustering cell first.')
if 'models' not in globals():
    raise RuntimeError('models not found. Please run the regression-mixture clustering cell first.')

CLUSTER1_ID = 1

mask_c1 = df_out['reg_cluster'] == CLUSTER1_ID
resid_c1 = df_out.loc[mask_c1, 'shift_value'].values - models[CLUSTER1_ID].predict(
    df_out.loc[mask_c1, 'shielding_value'].values.reshape(-1, 1)
)
sigma1 = float(np.std(resid_c1, ddof=1))
print(f'Estimated sigma for prioritized Cluster {CLUSTER1_ID}: {sigma1:.3f}')

K_SIGMA = 2
thr = K_SIGMA * sigma1
print(f'Priority threshold: |residual to the Cluster 1 line| <= {thr:.3f} (k={K_SIGMA})')

x_all = df_out['shielding_value'].values.reshape(-1, 1)
y_all = df_out['shift_value'].values
yhat_c1 = models[CLUSTER1_ID].predict(x_all)
resid_all_c1 = np.abs(y_all - yhat_c1)

df_priority = df_out.copy()
df_priority['priority_cluster'] = np.where(resid_all_c1 <= thr, 1, 0)
df_priority['residual_to_cluster1'] = resid_all_c1

n1 = int((df_priority['priority_cluster'] == 1).sum())
n0 = int((df_priority['priority_cluster'] == 0).sum())
print(f'Priority Cluster1 count: {n1} ({n1 / len(df_priority) * 100:.2f}%)')
print(f'Remaining Cluster0 count: {n0} ({n0 / len(df_priority) * 100:.2f}%)')

out_csv_name = f"priority_clustering_cluster1_band_k{str(K_SIGMA).replace('.', 'p')}.csv"
OUT_CSV = TAB_DIR / out_csv_name
df_priority.to_csv(OUT_CSV, index=False)
report_saved_paths([OUT_CSV], 'Saved clustering tables')

fig, ax = create_single_axis_figure('tall')

df0 = df_priority[df_priority['priority_cluster'] == 0]
df1 = df_priority[df_priority['priority_cluster'] == 1]
ax.scatter(df0['shielding_value'], df0['shift_value'], s=16, alpha=0.65, label='Cluster 0')
ax.scatter(df1['shielding_value'], df1['shift_value'], s=16, alpha=0.70, label='Cluster 1 (priority)')

x_line = np.linspace(df_priority['shielding_value'].min(), df_priority['shielding_value'].max(), 300).reshape(-1, 1)
y_line = models[CLUSTER1_ID].predict(x_line)
ax.plot(x_line.ravel(), y_line, linewidth=2.2, label='Cluster 1 regression line')
ax.fill_between(
    x_line.ravel(),
    y_line - thr,
    y_line + thr,
    alpha=0.22,
    linewidth=0.0,
    label=rf'Priority band ($\pm {K_SIGMA}\sigma$)'
)

ax.set_xlabel('Isotropic shielding constant (ppm)')
ax.set_ylabel(r'$^{19}$F NMR chemical shift (ppm)')
ax.set_title('Priority band assignment around the Cluster 1 regression line')
ax.legend(loc='best')
style_axes(ax)
finalize_figure(fig)

OUT_FIG = FIG_DIR / f"priority_clustering_cluster1_band_k{str(K_SIGMA).replace('.', 'p')}.png"
saved_paths = save_figure(fig, OUT_FIG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 3. cis/trans heuristic screening by priority cluster

Purpose: Summarize the prevalence of simple cis/trans heuristics across the priority-clustering labels.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 3. cis/trans heuristic screening by priority cluster
# Purpose: Summarize the prevalence of simple cis/trans heuristics across the priority-clustering labels.
# -----------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
TAB_DIR = RESULTS_DIR / 'tables'
TAB_DIR.mkdir(parents=True, exist_ok=True)

IN_CSV = TAB_DIR / 'priority_clustering_cluster1_band_k2.csv'
SMILES_COL = 'SMILES'
CLUSTER_COL = 'priority_cluster'

apply_plot_style()

def has_cis_trans(smiles: str):
    mol = Chem.MolFromSmiles(smiles)
    if mol is None:
        return None

    for bond in mol.GetBonds():
        if bond.GetBondType() == Chem.BondType.DOUBLE:
            a1, a2 = bond.GetBeginAtom(), bond.GetEndAtom()
            subs1 = [n for n in a1.GetNeighbors() if n.GetIdx() != a2.GetIdx()]
            subs2 = [n for n in a2.GetNeighbors() if n.GetIdx() != a1.GetIdx()]
            if subs1 and subs2:
                return True

    ring_info = mol.GetRingInfo()
    for ring in ring_info.AtomRings():
        if len(ring) == 6:
            ring_set = set(ring)
            n_sub = 0
            for atom_idx in ring:
                atom = mol.GetAtomWithIdx(atom_idx)
                if any(nbr.GetIdx() not in ring_set for nbr in atom.GetNeighbors()):
                    n_sub += 1
                    if n_sub >= 2:
                        return True

    return False

df = pd.read_csv(IN_CSV)
for col in (SMILES_COL, CLUSTER_COL):
    if col not in df.columns:
        raise ValueError(f"Missing column '{col}'. Available: {list(df.columns)}")

df = df[df[CLUSTER_COL].isin([0, 1])].copy()

counts = {0: {'total': 0, 'cis_trans': 0, 'invalid': 0}, 1: {'total': 0, 'cis_trans': 0, 'invalid': 0}}
for _, row in df.iterrows():
    c = int(row[CLUSTER_COL])
    smi = str(row[SMILES_COL])
    counts[c]['total'] += 1
    flag = has_cis_trans(smi)
    if flag is None:
        counts[c]['invalid'] += 1
    elif flag:
        counts[c]['cis_trans'] += 1

print_section('cis/trans screening by priority cluster')
print_key_value_rows([('Input table', IN_CSV.resolve())])
for c in (0, 1):
    total = counts[c]['total']
    cis = counts[c]['cis_trans']
    inv = counts[c]['invalid']
    pct = 0.0 if total == 0 else 100.0 * cis / total
    print(f'Cluster {c}: total={total}, cis/trans flagged={cis}, invalid_smiles={inv}, percent={pct:.2f}%')

labels = ['Cluster 0', 'Cluster 1']
values = []
for c in (0, 1):
    total = counts[c]['total']
    cis = counts[c]['cis_trans']
    values.append(0.0 if total == 0 else cis / total)

fig, ax = create_single_axis_figure('tall')
ax.bar(labels, values, color='#4C78A8')

max_v = max(values) if values else 0.0
ax.set_ylim(0, max(1.12 * max_v, 0.1))
ax.set_title('Estimated cis/trans prevalence by priority cluster')
ax.set_xlabel('Priority cluster label')
ax.set_ylabel('Fraction flagged (cis/trans)')
style_axes(ax, grid_axis='y')

for i, v in enumerate(values):
    offset = 0.02 * max(max_v, 0.1)
    ax.text(i, v + offset, f'{v * 100:.1f}%', ha='center', va='bottom', fontsize=10)

fig.tight_layout()
plt.show()

cluster1_df = df[df[CLUSTER_COL] == 1].copy()
cluster1_df = cluster1_df[['SMILES', 'shift_value', 'shielding_value']]
OUT_CLUSTER1 = TAB_DIR / 'priority_cluster1.csv'
cluster1_df.to_csv(OUT_CLUSTER1, index=False)
report_saved_paths([OUT_CLUSTER1], 'Saved Cluster 1 tables')


### 4. Shift distribution after Cluster 1 filtering

Purpose: Visualize the chemical-shift distribution of the retained Cluster 1 subset.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 4. Shift distribution after Cluster 1 filtering
# Purpose: Visualize the chemical-shift distribution of the retained Cluster 1 subset.
# -----------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROJECT_ROOT = Path('.').resolve()
SHIFT_CSV_PATH = PROJECT_ROOT / 'results' / 'tables' / 'priority_clustering_cluster1_band_k2.csv'
FIG_DIR = PROJECT_ROOT / 'results' / 'figures'
FIG_DIR.mkdir(parents=True, exist_ok=True)

apply_plot_style()

if not SHIFT_CSV_PATH.exists():
    raise FileNotFoundError(
        f'Shift CSV file not found:\n  {SHIFT_CSV_PATH}\n'
        'Please run the priority-band assignment cell before running this step.'
    )

df = pd.read_csv(SHIFT_CSV_PATH)
required_cols = {'SMILES', 'shift_value', 'shielding_value', 'priority_cluster'}
missing = required_cols - set(df.columns)
if missing:
    raise ValueError(f'Missing required columns: {missing}. Found columns: {list(df.columns)}')

df['SMILES'] = df['SMILES'].astype(str).str.strip()
df['shift_value'] = pd.to_numeric(df['shift_value'], errors='coerce')
df['shielding_value'] = pd.to_numeric(df['shielding_value'], errors='coerce')

n_before = len(df)
df = df[df['priority_cluster'] == 1].copy()
df = df.dropna(subset=['SMILES', 'shift_value', 'shielding_value']).reset_index(drop=True)
n_after = len(df)

print_section('Cluster 1 subset overview')
print_key_value_rows([
    ('Project root', PROJECT_ROOT),
    ('Input file', SHIFT_CSV_PATH),
    ('Rows before filtering', n_before),
    ('Rows retained for Cluster 1', n_after),
])
display(prepare_display_table(df.head()))

values = df['shift_value'].to_numpy()
bins = 30 if len(values) >= 200 else max(10, int(np.sqrt(len(values))))

fig_path = FIG_DIR / 'shift_distribution_cluster1.png'
fig, ax = create_single_axis_figure('single_wide')
ax.hist(values, bins=bins, edgecolor='black', linewidth=0.8)
ax.set_xlabel(r'$^{19}$F chemical shift (ppm)')
ax.set_ylabel('Count')
ax.set_title(r'Distribution of $^{19}$F chemical shifts for the Cluster 1 subset')
fig.tight_layout()

saved_paths = save_figure(fig, fig_path)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 5. Dataset cleaning for the Cluster 1 subset

Purpose: Apply the existing cleaning workflow to the subset selected for the Cluster 1 analysis.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 5. Dataset cleaning for the Cluster 1 subset
# Purpose: Apply the existing cleaning workflow to the subset selected for the Cluster 1 analysis.
# -----------------------------------------------------------------------------

from pathlib import Path

# ---------- Define output path ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

CLEANED_CSV_PATH = PROCESSED_DIR / "cleaned_shift_data_with_shield_cluster1.csv"

# ---------- Initial statistics ----------
n_initial = len(df)

# ---------- Remove duplicated entries ----------
# Duplicates are defined as identical SMILES + identical shift_value
n_dup = df.duplicated(subset=["SMILES", "shift_value"]).sum()

df_cleaned = df.drop_duplicates(subset=["SMILES", "shift_value"]).copy()

# ---------- Remove extreme shift values ----------
# According to project decision:
# Remove shift_value > 0 ppm and shift_value < -250 ppm
condition = (df_cleaned["shift_value"] <= 0) & (df_cleaned["shift_value"] >= -250)

n_out_of_range = (~condition).sum()

df_cleaned = df_cleaned[condition].reset_index(drop=True)

# ---------- Final statistics ----------
n_final = len(df_cleaned)

print_section("Data Cleaning Summary")
print_key_value_rows([
    ('Initial sample count', n_initial),
    ('Duplicate entries removed', n_dup),
    ('Out-of-range entries removed', n_out_of_range),
    ('Final sample count', n_final),
])

# ---------- Save cleaned dataset ----------
df_cleaned.to_csv(CLEANED_CSV_PATH, index=False)

report_saved_paths([CLEANED_CSV_PATH], 'Saved cleaned datasets')
display(prepare_display_table(df_cleaned.head()))


### 6. Molecular feature construction for the Cluster 1 subset

Purpose: Build the shielding-aware descriptor matrix used for the Cluster 1 modeling study.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 6. Molecular feature construction for the Cluster 1 subset
# Purpose: Build the shielding-aware descriptor matrix used for the Cluster 1 modeling study.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd

from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors
from rdkit import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# ---------- Input check ----------
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

if "PROJECT_ROOT" not in globals():
    PROJECT_ROOT = Path(".").resolve()

if "df_cleaned" not in globals():
    raise RuntimeError("df_cleaned not found. Please run Cell 8 (data cleaning) first.")

required_cols = {"SMILES", "shift_value", "shielding_value"}
missing_cols = required_cols - set(df_cleaned.columns)
if missing_cols:
    raise ValueError(f"df_cleaned missing columns: {missing_cols}")

# Ensure numeric (drop rows with missing values)
df_cleaned = df_cleaned.copy()
df_cleaned["shift_value"] = pd.to_numeric(df_cleaned["shift_value"], errors="coerce")
df_cleaned["shielding_value"] = pd.to_numeric(df_cleaned["shielding_value"], errors="coerce")
df_cleaned = df_cleaned.dropna(subset=["shift_value", "shielding_value"]).reset_index(drop=True)

# ---------- Paths ----------
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

OUT_CSV = PROCESSED_DIR / "features_dataset_with_shield_cluster1.csv"
OUT_PARQUET = PROCESSED_DIR / "features_dataset_with_shield_cluster1.parquet"

# ---------- Parameters ----------
N_BITS = 1024
MORGAN_RADIUS = 2
ECFP_RADIUS = 4

# ---------- Fingerprint generators (RDKit-compatible signature) ----------
# IMPORTANT:
# Your RDKit build supports includeChirality, but does NOT accept includeFeatures.
morgan_gen = GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=N_BITS, includeChirality=False)
ecfp_gen = GetMorganGenerator(radius=ECFP_RADIUS, fpSize=N_BITS, includeChirality=True)

# =========================
# Helper functions
# =========================

def mol_from_smiles(smiles: str):
    """Convert SMILES to RDKit Mol object. Returns None if parsing fails."""
    try:
        return Chem.MolFromSmiles(smiles)
    except Exception:
        return None

def bitvect_to_numpy(fp) -> np.ndarray:
    """Convert RDKit ExplicitBitVect to a numpy array of 0/1."""
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def compute_basic_descriptors(mol: Chem.Mol) -> dict:
    """Compute basic physicochemical descriptors (numeric)."""
    return {
        "MolWt": Descriptors.MolWt(mol),
        "LogP": Descriptors.MolLogP(mol),
        "TPSA": Descriptors.TPSA(mol),
        "NumHDonors": Descriptors.NumHDonors(mol),
        "NumHAcceptors": Descriptors.NumHAcceptors(mol),
        "NumRotatableBonds": Descriptors.NumRotatableBonds(mol),
        "RingCount": Descriptors.RingCount(mol),
    }

def compute_morgan_fp(mol: Chem.Mol) -> np.ndarray:
    """Compute Morgan fingerprint (1024 bits) using MorganGenerator."""
    fp = morgan_gen.GetFingerprint(mol)
    return bitvect_to_numpy(fp)

def compute_ecfp_per_fluorine_or(mol: Chem.Mol) -> np.ndarray:
    """
    Compute fluorine-centered ECFP-like fingerprints (1024 bits).
    For multiple F atoms, combine fingerprints using bitwise OR (max).

    NOTE:
    This uses MorganGenerator with fromAtoms selection.
    It does NOT enable 'feature Morgan' to keep compatibility with your RDKit build.
    """
    fluorines = [a for a in mol.GetAtoms() if a.GetSymbol() == "F"]
    if not fluorines:
        return np.zeros((N_BITS,), dtype=np.int8)

    combined = np.zeros((N_BITS,), dtype=np.int8)
    for atom in fluorines:
        fp = ecfp_gen.GetFingerprint(mol, fromAtoms=[atom.GetIdx()])
        combined = np.maximum(combined, bitvect_to_numpy(fp))
    return combined

def compute_maccs_fp(mol: Chem.Mol) -> np.ndarray:
    """
    Compute MACCS keys.
    RDKit typically returns 167 bits (including bit 0). Keep native length.
    """
    fp = MACCSkeys.GenMACCSKeys(mol)
    return bitvect_to_numpy(fp)

# =========================
# Feature construction
# =========================

feature_rows = []
invalid_smiles_count = 0

for _, row in df_cleaned.iterrows():
    smiles = str(row["SMILES"]).strip()
    shift = float(row["shift_value"])
    shield = float(row["shielding_value"])

    mol = mol_from_smiles(smiles)
    if mol is None:
        invalid_smiles_count += 1
        continue

    basic_desc = compute_basic_descriptors(mol)
    morgan_fp = compute_morgan_fp(mol)
    ecfp_f_fp = compute_ecfp_per_fluorine_or(mol)
    maccs_fp = compute_maccs_fp(mol)

    # Core columns
    feature_dict = {
        "SMILES": smiles,
        "shift_value": shift,
        "shielding_value": shield,
        **basic_desc,
    }

    # Morgan bits
    for i, bit in enumerate(morgan_fp):
        feature_dict[f"Morgan_{i}"] = int(bit)

    # F-centered ECFP-like bits
    for i, bit in enumerate(ecfp_f_fp):
        feature_dict[f"ECFP_F_{i}"] = int(bit)

    # MACCS bits (native length)
    for i, bit in enumerate(maccs_fp):
        feature_dict[f"MACCS_{i}"] = int(bit)

    feature_rows.append(feature_dict)

df_features = pd.DataFrame(feature_rows)

# =========================
# Save dataset (CSV required; Parquet optional)
# =========================

df_features.to_csv(OUT_CSV, index=False)

try:
    df_features.to_parquet(OUT_PARQUET, index=False)
    parquet_msg = f"Saved Parquet to: {OUT_PARQUET}"
except Exception as e:
    parquet_msg = f"Parquet not saved (optional). Reason: {type(e).__name__}: {e}"

# =========================
# Summary
# =========================

n_desc = 7
n_morgan = N_BITS
n_ecfp = N_BITS
n_maccs = len([c for c in df_features.columns if c.startswith("MACCS_")])

print_section("Feature Engineering Summary (with shielding_value)")
print(f"Total input samples (df_cleaned) : {len(df_cleaned)}")
print(f"Invalid SMILES removed           : {invalid_smiles_count}")
print(f"Final dataset size               : {len(df_features)}")

print("\n----- Feature dimensions -----")
print(f"shielding_value                  : 1")
print(f"PhysChem descriptors             : {n_desc}")
print(f"Morgan fingerprint               : {n_morgan}")
print(f"Fluorine-centered ECFP-like (OR) : {n_ecfp}")
print(f"MACCS keys (native length)       : {n_maccs}")
print(f"Total feature dimension          : {df_features.shape[1] - 2}")  # excluding SMILES & shift_value

print("\n----- Output files -----")
print(f"Saved CSV to                     : {OUT_CSV}")
print_section('Saved feature files')
print(parquet_msg)

display(prepare_display_table(df_features.head()))


### 7. Residual-learning benchmark for the Cluster 1 subset

Purpose: Compare residual-learning models after removing the linear shielding baseline.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 7. Residual-learning benchmark for the Cluster 1 subset
# Purpose: Compare residual-learning models after removing the linear shielding baseline.
# -----------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression, Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
TABLE_DIR = RESULTS_DIR / 'tables'
FIG_DIR = RESULTS_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_TABLE = TABLE_DIR / 'residual_model_benchmark_results.csv'
OUT_FIG = FIG_DIR / 'residual_model_benchmark_mae.png'

apply_plot_style()

df_features = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'features_dataset_with_shield_cluster1.parquet')

X_linear = df_features[['shielding_value']]
y = df_features['shift_value'].astype(float)
lin_model = LinearRegression()
lin_model.fit(X_linear, y)
pre_shift = lin_model.predict(X_linear)
residual = y - pre_shift

print_section('Stage 1 Linear Model')
print(f'Slope      : {lin_model.coef_[0]:.5f}')
print(f'Intercept  : {lin_model.intercept_:.5f}')
print(f'Baseline MAE: {mean_absolute_error(y, pre_shift):.4f}')

X = df_features.drop(columns=['SMILES', 'shift_value', 'shielding_value'])
print_section('Residual Dataset Summary')
print(f'Total samples : {len(X)}')
print(f'Feature dim   : {X.shape[1]}')
print(f"shielding_value included in X? : {'shielding_value' in X.columns}")

X_train, X_test, r_train, r_test, pre_train, pre_test = train_test_split(X, residual, pre_shift, test_size=0.1, random_state=SEED)
print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')

models = {
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'DecisionTree': DecisionTreeRegressor(random_state=SEED),
    'RandomForest': RandomForestRegressor(random_state=SEED),
    'ExtraTrees': ExtraTreesRegressor(random_state=SEED),
    'GradientBoosting': GradientBoostingRegressor(random_state=SEED),
    'AdaBoost': AdaBoostRegressor(random_state=SEED),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(),
}

results = []
for name, model in models.items():
    try:
        model.fit(X_train, r_train)
        r_pred = model.predict(X_test)
        shift_pred = pre_test + r_pred
        shift_true = pre_test + r_test

        mae = mean_absolute_error(shift_true, shift_pred)
        rmse = float(np.sqrt(mean_squared_error(shift_true, shift_pred)))
        r2 = r2_score(shift_true, shift_pred)
        results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
        print(metric_row(name, mae, rmse, r2))
    except Exception as e:
        print(f'{name} failed: {type(e).__name__}: {e}')

df_results = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
df_results.to_csv(OUT_TABLE, index=False)

report_saved_paths([OUT_TABLE], 'Saved result tables')
display(prepare_display_table(df_results))

fig, ax = create_single_axis_figure('single_wide')
ax.bar(df_results['Model'], df_results['MAE'], color='#4C78A8')
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel('MAE (ppm)')
ax.set_title('Benchmark of residual-learning models')
style_axes(ax, grid_axis='y')
fig.tight_layout()

saved_paths = save_figure(fig, OUT_FIG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 8. Direct-regression benchmark for the Cluster 1 subset

Purpose: Benchmark direct regression models on the Cluster 1 feature set without the residual-learning decomposition.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 8. Direct-regression benchmark for the Cluster 1 subset
# Purpose: Benchmark direct regression models on the Cluster 1 feature set without the residual-learning decomposition.
# -----------------------------------------------------------------------------

from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import Ridge, Lasso, ElasticNet
from sklearn.ensemble import RandomForestRegressor, GradientBoostingRegressor, ExtraTreesRegressor, AdaBoostRegressor
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
TABLE_DIR = RESULTS_DIR / 'tables'
FIG_DIR = RESULTS_DIR / 'figures'
TABLE_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

OUT_TABLE = TABLE_DIR / 'direct_model_benchmark_results.csv'
OUT_FIG = FIG_DIR / 'direct_model_benchmark_mae.png'

apply_plot_style()

df_features = pd.read_parquet(PROJECT_ROOT / 'data' / 'processed' / 'features_dataset_with_shield_cluster1.parquet')
X = df_features.drop(columns=['SMILES', 'shift_value'])
y = df_features['shift_value'].astype(float)

print_section('Dataset Summary')
print(f'Total samples : {len(X)}')
print(f'Feature dim   : {X.shape[1]}')
print(f"shielding_value included in X? : {'shielding_value' in X.columns}")

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.1, random_state=SEED)
print(f'Train samples : {len(X_train)}')
print(f'Test samples  : {len(X_test)}')

models = {
    'Ridge': Ridge(),
    'Lasso': Lasso(),
    'ElasticNet': ElasticNet(),
    'DecisionTree': DecisionTreeRegressor(random_state=SEED),
    'RandomForest': RandomForestRegressor(random_state=SEED),
    'ExtraTrees': ExtraTreesRegressor(random_state=SEED),
    'GradientBoosting': GradientBoostingRegressor(random_state=SEED),
    'AdaBoost': AdaBoostRegressor(random_state=SEED),
    'SVR': SVR(),
    'KNN': KNeighborsRegressor(),
}

results = []
for name, model in models.items():
    try:
        model.fit(X_train, y_train)
        y_pred = model.predict(X_test)
        mae = mean_absolute_error(y_test, y_pred)
        rmse = float(np.sqrt(mean_squared_error(y_test, y_pred)))
        r2 = r2_score(y_test, y_pred)
        results.append({'Model': name, 'MAE': mae, 'RMSE': rmse, 'R2': r2})
        print(metric_row(name, mae, rmse, r2))
    except Exception as e:
        print(f'{name} failed: {type(e).__name__}: {e}')

df_results = pd.DataFrame(results).sort_values('MAE').reset_index(drop=True)
df_results.to_csv(OUT_TABLE, index=False)

report_saved_paths([OUT_TABLE], 'Saved result tables')
display(prepare_display_table(df_results))

fig, ax = create_single_axis_figure('single_wide')
ax.bar(df_results['Model'], df_results['MAE'], color='#4C78A8')
ax.tick_params(axis='x', rotation=90)
ax.set_ylabel('MAE (ppm)')
ax.set_title('Benchmark of direct regression models')
style_axes(ax, grid_axis='y')
fig.tight_layout()

saved_paths = save_figure(fig, OUT_FIG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 9. Train, validation, and test parity analysis for the Cluster 1 subset

Purpose: Visualize split-wise predictive agreement for the selected Cluster 1 model.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 9. Train, validation, and test parity analysis for the Cluster 1 subset
# Purpose: Visualize split-wise predictive agreement for the selected Cluster 1 model.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
PROJECT_ROOT = Path('.').resolve()
RESULTS_DIR = PROJECT_ROOT / 'results'
FIG_DIR = RESULTS_DIR / 'figures'
MODEL_DIR = RESULTS_DIR / 'models'
FIG_DIR.mkdir(parents=True, exist_ok=True)

FEATURE_PATH = PROJECT_ROOT / 'data' / 'processed' / 'features_dataset_with_shield_cluster1.parquet'
if not FEATURE_PATH.exists():
    raise FileNotFoundError(f'Feature dataset not found: {FEATURE_PATH}')

MODEL_PATH = MODEL_DIR / 'best_on_val_refined_ExtraTrees_cluster1.pkl'
if not MODEL_PATH.exists():
    raise FileNotFoundError(f'Model not found: {MODEL_PATH}')

OUT_FIG = FIG_DIR / 'best_on_val_refined_ExtraTrees_cluster1.png'

apply_plot_style()

df_features = pd.read_parquet(FEATURE_PATH)
required_cols = {'SMILES', 'shift_value', 'shielding_value'}
missing = required_cols - set(df_features.columns)
if missing:
    raise ValueError(f'Feature dataset missing required columns: {missing}')

X = df_features.drop(columns=['SMILES', 'shift_value'])
y = df_features['shift_value'].astype(float)

print_section('Dataset loaded')
print(f'Samples      : {len(df_features)}')
print(f'Feature dim  : {X.shape[1]}')
print(f'Feature file : {FEATURE_PATH.resolve()}')
print(f'Model file   : {MODEL_PATH.resolve()}')

X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=SEED)
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1 / 9), random_state=SEED)

print_section('Split Summary')
print(f'Train samples : {len(X_train)}')
print(f'Val samples   : {len(X_val)}')
print(f'Test samples  : {len(X_test)}')

def regression_metrics(y_true, y_pred):
    mae = mean_absolute_error(y_true, y_pred)
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = r2_score(y_true, y_pred)
    return mae, rmse, r2

def parity_plot(ax, y_true, y_pred, title):
    ax.scatter(y_true, y_pred, s=18, alpha=0.75, edgecolor='none')
    y_min, y_max = compute_plot_limits(y_true, y_pred, pad_ratio=0.0)
    add_identity_line(ax, y_min, y_max)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    ax.set_xlim(y_min, y_max)
    ax.set_ylim(y_min, y_max)
    style_axes(ax, equal_aspect=True)

loaded = joblib.load(MODEL_PATH)
if isinstance(loaded, dict):
    for _k in ['model', 'best_model', 'best_estimator', 'best_estimator_', 'estimator']:
        if _k in loaded:
            model = loaded[_k]
            break
    else:
        raise TypeError(f'Loaded object is a dict but no known model key found. Available keys: {list(loaded.keys())}')
else:
    model = loaded

y_pred_train = model.predict(X_train)
y_pred_val = model.predict(X_val)
y_pred_test = model.predict(X_test)

train_mae, train_rmse, train_r2 = regression_metrics(y_train, y_pred_train)
val_mae, val_rmse, val_r2 = regression_metrics(y_val, y_pred_val)
test_mae, test_rmse, test_r2 = regression_metrics(y_test, y_pred_test)

print_section('Performance')
print(metric_row('Train', train_mae, train_rmse, train_r2, label_width=10))
print(metric_row('Validation', val_mae, val_rmse, val_r2, label_width=12))
print(metric_row('Test', test_mae, test_rmse, test_r2, label_width=12))

fig, axes = create_figure('triptych', nrows=1, ncols=3)
ax1, ax2, ax3 = axes

parity_plot(ax1, y_train.values, y_pred_train, 'Train')
annotate_panel_text(ax1, metric_text(train_mae, train_rmse, train_r2, decimals=2))
parity_plot(ax2, y_val.values, y_pred_val, 'Validation')
annotate_panel_text(ax2, metric_text(val_mae, val_rmse, val_r2, decimals=2))
parity_plot(ax3, y_test.values, y_pred_test, 'Test')
annotate_panel_text(ax3, metric_text(test_mae, test_rmse, test_r2, decimals=2))

fig.suptitle(f'Parity plots for the Cluster 1 ExtraTrees model: {MODEL_PATH.stem}', fontsize=13)
fig.tight_layout()

saved_paths = save_figure(fig, OUT_FIG)
plt.show()

report_saved_paths(saved_paths, 'Saved parity-plot files')


### 10. Compare ExtraTrees Performance Before and After Cluster 1 Cleaning

Purpose: Compare the shielding-aware ExtraTrees model before and after applying the Cluster 1 cleaning strategy.


In [ ]:
# -----------------------------------------------------------------------------
# Compare ExtraTrees performance before and after Cluster 1 cleaning
# Purpose: Evaluate the saved shielding-aware ExtraTrees models on the two feature sets and export the parity comparison plot.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
FEATURE_BEFORE = Path('data/processed/features_dataset_with_shield.parquet')
FEATURE_AFTER = Path('data/processed/features_dataset_with_shield_cluster1.parquet')
MODEL_BEFORE = Path('results/models/best_on_val_refined_ExtraTrees_with_shield.pkl')
MODEL_AFTER = Path('results/models/best_on_val_refined_ExtraTrees_cluster1.pkl')

OUT_PNG = Path('results/figures/Figure4_ExtraTrees_Cleaning_Test_AB.png')
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

apply_plot_style()

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

def load_xy_predict(feature_path, model_path):
    df = pd.read_parquet(feature_path)
    X = df.drop(columns=['SMILES', 'shift_value'])
    y = df['shift_value'].astype(float)
    X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.10, random_state=SEED)
    X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=(1 / 9), random_state=SEED)

    loaded = joblib.load(model_path)
    if isinstance(loaded, dict):
        for k in ['model', 'best_model', 'best_estimator', 'best_estimator_', 'estimator']:
            if k in loaded:
                loaded = loaded[k]
                break
        else:
            raise TypeError(f'Loaded object is a dict but no known model key found. Available keys: {list(loaded.keys())}')

    y_pred_test = loaded.predict(X_test)
    mae, rmse, r2 = regression_metrics(y_test, y_pred_test)
    return y_test.values, y_pred_test, (mae, rmse, r2)

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)
    annotate_panel_text(ax, metric_text(mae, rmse, r2, decimals=2))

y_true_A, y_pred_A, mA = load_xy_predict(FEATURE_BEFORE, MODEL_BEFORE)
y_true_B, y_pred_B, mB = load_xy_predict(FEATURE_AFTER, MODEL_AFTER)

vmin, vmax = compute_plot_limits(y_true_A, y_pred_A, y_true_B, y_pred_B)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true_A, y_pred_A, 'ExtraTrees with shielding (test)', mA, vmin, vmax)
parity(axes[1], y_true_B, y_pred_B, 'ExtraTrees with shielding (Cluster 1 test)', mB, vmin, vmax)

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 11. Evaluate External Transfer With Linear Calibration

Purpose: Apply the shielding-aware ExtraTrees model to the 60 MHz dataset and visualize the effect of linear calibration.


In [ ]:
# -----------------------------------------------------------------------------
# Evaluate external transfer with linear calibration
# Purpose: Load the external 60 MHz feature set, apply the pretrained model, fit a linear calibration, and export the before-and-after parity plots.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib
import matplotlib.pyplot as plt

from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.linear_model import LinearRegression

from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

SEED = 42
LAB_FEATURES = Path('data/processed/features_dataset_60M_NMR.parquet')
PRETRAINED_MODEL = Path('results/models/best_on_val_refined_ExtraTrees_with_shield.pkl')

OUT_PNG = Path('results/figures/Figure_LabTransfer_LinearCalibration_AB.png')
OUT_PNG.parent.mkdir(parents=True, exist_ok=True)

SMILES_COL = 'SMILES'
TARGET_COL = 'shift_value'

apply_plot_style()

def regression_metrics(y_true, y_pred):
    mae = float(mean_absolute_error(y_true, y_pred))
    rmse = float(np.sqrt(mean_squared_error(y_true, y_pred)))
    r2 = float(r2_score(y_true, y_pred))
    return mae, rmse, r2

def load_model(model_path: Path):
    loaded = joblib.load(model_path)
    if isinstance(loaded, dict):
        for k in ['model', 'best_model', 'best_estimator', 'best_estimator_', 'estimator']:
            if k in loaded:
                loaded = loaded[k]
                break
        else:
            raise TypeError(f'Loaded object is a dict but no known model key found. Available keys: {list(loaded.keys())}')
    return loaded

def parity(ax, y_true, y_pred, title, metrics, vmin, vmax, extra_text=None):
    mae, rmse, r2 = metrics
    ax.scatter(y_true, y_pred, s=18, alpha=0.80, edgecolor='none')
    add_identity_line(ax, vmin, vmax)
    ax.set_xlim(vmin, vmax)
    ax.set_ylim(vmin, vmax)
    ax.set_title(title)
    ax.set_xlabel('True lab shift (ppm)')
    ax.set_ylabel('Predicted shift (ppm)')
    style_axes(ax, equal_aspect=True)

    text = metric_text(mae, rmse, r2, decimals=2)
    if extra_text:
        text = extra_text + '\n' + text
    annotate_panel_text(ax, text)

df = pd.read_parquet(LAB_FEATURES)
if SMILES_COL not in df.columns:
    raise KeyError(f'Missing column: {SMILES_COL}')
if TARGET_COL not in df.columns:
    raise KeyError(f'Missing column: {TARGET_COL}')

smiles = df[SMILES_COL].astype(str).values
X = df.drop(columns=[SMILES_COL, TARGET_COL])
y_true = df[TARGET_COL].astype(float).values

model = load_model(PRETRAINED_MODEL)
y_pred_raw = model.predict(X)

lin = LinearRegression(fit_intercept=True)
lin.fit(y_pred_raw.reshape(-1, 1), y_true)
a = float(lin.coef_[0])
b = float(lin.intercept_)

y_pred_cal = a * y_pred_raw + b
m_raw = regression_metrics(y_true, y_pred_raw)
m_cal = regression_metrics(y_true, y_pred_cal)

err_raw = y_pred_raw - y_true
err_cal = y_pred_cal - y_true
err_table = pd.DataFrame({
    'SMILES': smiles,
    'True_shift_ppm': y_true,
    'Pred_raw_ppm': y_pred_raw,
    'Pred_cal_ppm': y_pred_cal,
    'Err_raw_ppm': err_raw,
    'AbsErr_raw_ppm': np.abs(err_raw),
    'Err_cal_ppm': err_cal,
    'AbsErr_cal_ppm': np.abs(err_cal),
}).sort_values('AbsErr_cal_ppm', ascending=False).reset_index(drop=True)

with pd.option_context('display.max_rows', 200, 'display.max_columns', 50, 'display.width', 200):
    print_section('Per-molecule error table (sorted by AbsErr_cal)')
    print(prepare_display_table(err_table))

vmin, vmax = compute_plot_limits(y_true, y_pred_raw, y_pred_cal)
fig, axes = create_figure('comparison', nrows=1, ncols=2, constrained_layout=True)
parity(axes[0], y_true, y_pred_raw, 'ExtraTrees -> Lab', m_raw, vmin, vmax)
parity(axes[1], y_true, y_pred_cal, 'ExtraTrees -> Lab + linear calibration', m_cal, vmin, vmax, extra_text=f'y = {a:.3f} * yhat + {b:.3f}')

saved_paths = save_figure(fig, OUT_PNG)
plt.show()

report_saved_paths(saved_paths, 'Saved figure files')


### 12. Calibrated prediction utilities for external molecules

Purpose: Collect the reusable inference utilities required for calibrated single-molecule and batch prediction.

Presentation note: the code below preserves the original analytical logic while using the standardized project plotting and reporting style.


In [ ]:
# -----------------------------------------------------------------------------
# 12. Calibrated prediction utilities for external molecules
# Purpose: Collect the reusable inference utilities required for calibrated single-molecule and batch prediction.
# -----------------------------------------------------------------------------

from pathlib import Path
import numpy as np
import pandas as pd
import joblib

from rdkit import Chem
from rdkit.Chem import MACCSkeys, Descriptors
from rdkit import DataStructs
from rdkit.Chem.rdFingerprintGenerator import GetMorganGenerator

# =========================
# Config
# =========================
from plotting_standard import add_identity_line, add_panel_label, annotate_bar_values, annotate_panel_text, apply_plot_style, compute_plot_limits, create_figure, create_single_axis_figure, finalize_figure, metric_row, metric_text, prepare_display_table, print_key_value_rows, print_section, report_saved_paths, save_figure, style_axes

PRETRAINED_MODEL = Path('results/models/best_on_val_refined_ExtraTrees_with_shield.pkl')

# This file is used only to recover the training feature columns and must match the original model inputs.
REFERENCE_FEATURES = Path('data/processed/features_dataset_with_shield.parquet')

# Fixed linear calibration: y = 0.904 * yhat - 6.457
CALIB_A = 0.904
CALIB_B = -6.457

# Feature parameters (must match training / Cell 8)
N_BITS = 1024
MORGAN_RADIUS = 2
ECFP_RADIUS = 4

morgan_gen = GetMorganGenerator(radius=MORGAN_RADIUS, fpSize=N_BITS, includeChirality=False)
ecfp_gen = GetMorganGenerator(radius=ECFP_RADIUS, fpSize=N_BITS, includeChirality=True)

def load_model(model_path: Path):
    loaded = joblib.load(model_path)
    if isinstance(loaded, dict):
        for k in ['model', 'best_model', 'best_estimator', 'best_estimator_', 'estimator']:
            if k in loaded:
                loaded = loaded[k]
                break
        else:
            raise TypeError(
                f'Loaded object is a dict but no known model key found. '
                f'Available keys: {list(loaded.keys())}'
            )
    return loaded

def mol_from_smiles(smiles: str):
    try:
        return Chem.MolFromSmiles(str(smiles).strip())
    except Exception:
        return None

def bitvect_to_numpy(fp) -> np.ndarray:
    arr = np.zeros((fp.GetNumBits(),), dtype=np.int8)
    DataStructs.ConvertToNumpyArray(fp, arr)
    return arr

def compute_basic_descriptors(mol: Chem.Mol) -> dict:
    return {
        'MolWt': Descriptors.MolWt(mol),
        'LogP': Descriptors.MolLogP(mol),
        'TPSA': Descriptors.TPSA(mol),
        'NumHDonors': Descriptors.NumHDonors(mol),
        'NumHAcceptors': Descriptors.NumHAcceptors(mol),
        'NumRotatableBonds': Descriptors.NumRotatableBonds(mol),
        'RingCount': Descriptors.RingCount(mol),
    }

def compute_morgan_fp(mol: Chem.Mol) -> np.ndarray:
    fp = morgan_gen.GetFingerprint(mol)
    return bitvect_to_numpy(fp)

def compute_ecfp_per_fluorine_or(mol: Chem.Mol) -> np.ndarray:
    fluorines = [a for a in mol.GetAtoms() if a.GetSymbol() == 'F']
    if not fluorines:
        return np.zeros((N_BITS,), dtype=np.int8)

    combined = np.zeros((N_BITS,), dtype=np.int8)
    for atom in fluorines:
        fp = ecfp_gen.GetFingerprint(mol, fromAtoms=[atom.GetIdx()])
        combined = np.maximum(combined, bitvect_to_numpy(fp))
    return combined

def compute_maccs_fp(mol: Chem.Mol) -> np.ndarray:
    fp = MACCSkeys.GenMACCSKeys(mol)
    return bitvect_to_numpy(fp)

def get_reference_feature_columns(ref_path: Path, smiles_col='SMILES', target_col='shift_value'):
    if ref_path.suffix.lower() == '.parquet':
        cols = pd.read_parquet(ref_path).columns.tolist()
    elif ref_path.suffix.lower() == '.csv':
        cols = pd.read_csv(ref_path, nrows=1).columns.tolist()
    else:
        raise ValueError(f'Unsupported reference feature file type: {ref_path}')

    drop_cols = [c for c in [smiles_col, target_col] if c in cols]
    feat_cols = [c for c in cols if c not in drop_cols]
    return feat_cols

def align_to_reference_columns(df_feat: pd.DataFrame, ref_feature_cols: list) -> pd.DataFrame:
    df = df_feat.copy()
    if 'SMILES' in df.columns:
        df = df.drop(columns=['SMILES'])

    missing = [c for c in ref_feature_cols if c not in df.columns]
    for c in missing:
        df[c] = 0

    extra = [c for c in df.columns if c not in ref_feature_cols]
    if extra:
        df = df.drop(columns=extra)

    return df[ref_feature_cols]

def featurize_one(smiles: str, shielding_value: float) -> pd.DataFrame:
    smiles = str(smiles).strip()
    mol = mol_from_smiles(smiles)
    if mol is None:
        raise ValueError(f'Invalid SMILES: {smiles}')

    shield = float(shielding_value)
    basic_desc = compute_basic_descriptors(mol)
    morgan_fp = compute_morgan_fp(mol)
    ecfp_f_fp = compute_ecfp_per_fluorine_or(mol)
    maccs_fp = compute_maccs_fp(mol)

    feature_dict = {'SMILES': smiles, 'shielding_value': shield, **basic_desc}
    for i, bit in enumerate(morgan_fp):
        feature_dict[f'Morgan_{i}'] = int(bit)
    for i, bit in enumerate(ecfp_f_fp):
        feature_dict[f'ECFP_F_{i}'] = int(bit)
    for i, bit in enumerate(maccs_fp):
        feature_dict[f'MACCS_{i}'] = int(bit)

    return pd.DataFrame([feature_dict])

class ShiftPredictor:
    def __init__(self, model_path: Path, reference_features_path: Path, calib_a: float, calib_b: float):
        self.model = load_model(model_path)
        self.ref_cols = get_reference_feature_columns(reference_features_path)
        self.a = float(calib_a)
        self.b = float(calib_b)

    def predict_one(self, smiles: str, shielding_value: float) -> dict:
        df1 = featurize_one(smiles, shielding_value)
        X1 = align_to_reference_columns(df1, self.ref_cols)
        y_raw = float(self.model.predict(X1)[0])
        y_cal = float(self.a * y_raw + self.b)
        return {
            'SMILES': str(smiles).strip(),
            'shielding_value': float(shielding_value),
            'pred_shift_raw': y_raw,
            'pred_shift_calibrated': y_cal,
        }

    def predict_df(self, df_in: pd.DataFrame, smiles_col='SMILES', shield_col='shielding_value') -> pd.DataFrame:
        rows = []
        for _, r in df_in.iterrows():
            rows.append(self.predict_one(r[smiles_col], r[shield_col]))
        return pd.DataFrame(rows)

if __name__ == '__main__':
    predictor = ShiftPredictor(
        model_path=PRETRAINED_MODEL,
        reference_features_path=REFERENCE_FEATURES,
        calib_a=CALIB_A,
        calib_b=CALIB_B,
    )

    # Single-molecule prediction example
    # out = predictor.predict_one('C1=CC(=CC=C1C(F)(F)F)Br', 272.74)
    # print(out)

    # Batch prediction example
    df_new = pd.DataFrame({
        'SMILES': [
            'FC(F)(F)C1=CC=C(C=C1)N2CC=3C=CC=CC3CC2',
            'O=C(N(C)CC1=CC=C(C=C1)C(F)(F)F)C',
            'FC(F)(F)C1=CC=C(Cl)C=C1',
            'FC(F)(F)C=1C=CC=CC1',
            'FC(F)(F)C1=CC=C(Br)C=C1',
        ],
        'shielding_value': [261.74, 266.78, 270.54, 269.02, 273.12],
    })
    print(predictor.predict_df(df_new))
